[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/yildiztu/Code-101-Problems-Logistics-SCM/blob/main/Part%20II%20-%20The%20End-to-End%20Supply%20Chain%20%28Problems%2047-101%29/E.%20Integrated%2C%20Resilient%2C%20%26%20Modern%20Supply%20Chains%20%28The%20Frontier%29/098.%20The%20Green%20Vehicle%20Routing%20Problem/P98-Tier-7_executed.ipynb)

---

# 98. The Green Vehicle Routing Problem

## Tier 7: Green Dispatch Advisor

### Purpose

This notebook demonstrates a human–AI decision-support workflow for a small green-routing example. An explainable AI component evaluates a supplied set of routes, presents cost and emissions information, and records a human dispatcher's decision. The dispatcher retains final authority and can consider contextual information that is not represented in the route metrics.

### Inputs and outputs

**Inputs:** a distance matrix, an emissions matrix, two prescribed vehicle routes, customer-context records, and a set of preference weights used to label recommendation profiles.

**Outputs:** route cost and emissions metrics, explanations and contextual notes, a recorded dispatcher decision, a session summary, and visualizations of the candidate recommendations and decision history.

### Scope and modeling assumptions

- The routes are supplied as fixed input. This notebook evaluates them; it does **not** search for or optimize routes.
- Every recommendation profile uses the same routes. Consequently, all profiles have the same physical distance, cost, and emissions. The profiles communicate different priorities; they are not distinct routing solutions.
- Operating cost is modeled as a constant rate of \$2 per kilometer. The emissions matrix supplies emissions in kilograms of CO₂ for each traversed arc.
- Customer requirements, traffic, weather, time windows, vehicle capacity, and vehicle technology are stored or discussed as context, but they do not alter route feasibility, cost, or emissions in this demonstration.
- The learning component records dispatcher choices. It does not update model parameters or change future recommendations. Agreement rate and decision quality are not calculated because the example defines no independent AI choice or outcome-based quality measure.

### Mathematical representation

Let a route arc be an ordered pair of consecutive nodes in the supplied routes. The total distance and emissions are

$$D = \sum_{r} \sum_{(i,j)\in r} d_{ij}, \qquad E = \sum_{r} \sum_{(i,j)\in r} e_{ij}, \qquad C = 2D.$$

Here, $r$ indexes the prescribed routes, $d_{ij}$ is the distance in kilometers from node $i$ to node $j$, $e_{ij}$ is the corresponding emissions quantity in kilograms of CO₂, and $C$ is the modeled operating cost in dollars. The factor 2 is the assumed cost rate of \$2 per kilometer. In the code, these calculations are performed in `ExplainableAI.generate_pareto_plans`.

For two objectives to minimize, a candidate $p$ dominates candidate $q$ if $C_p \leq C_q$ and $E_p \leq E_q$, with at least one strict inequality. A Pareto frontier contains candidates that are not dominated by another candidate. Since the candidates here share the same routes, they share the same objective values: there is one unique cost–emissions outcome, not a cost–emissions trade-off. The visualization makes this limitation visible rather than implying that different routes were optimized.

### Human–AI decision workflow

1. Evaluate the prescribed routes and create recommendation profiles for cost-focused, balanced, and emissions-focused preferences.
2. Show the dispatcher each profile's metrics, explanation, and relevant customer-context notes.
3. Apply the dispatcher's stated preference to select a profile. Context is available to the dispatcher, but does not change the computed route metrics.
4. Record the decision, confidence, and decision time. Summarize the observed sessions without presenting simulated indicators as measured outcomes.

### What to look for

- The candidate profiles occupy the same point in cost–emissions space because the route input is fixed.
- Explanation text distinguishes the profiles' stated priorities from the physical metrics actually calculated.
- Customer context can inform a human decision even though it is not a routing constraint in this model.
- Confidence and decision time are recorded for the example sessions; they should not be interpreted as evidence of trust, decision quality, or learning effectiveness.

### Why human–AI decision support is useful

Routing decisions can depend on operational knowledge that is absent from a mathematical model. An AI component can calculate and communicate defined metrics consistently, while a human dispatcher can apply contextual judgment and remain accountable for the final choice. A useful decision-support system should make clear which factors were calculated, which were supplied as context, and which were left outside the model.

In [ ]:
from dataclasses import dataclass
from datetime import datetime
import random
from typing import Dict, List, Optional, Tuple

import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns


plt.style.use("seaborn-v0_8")
sns.set_palette("husl")

# Fixed seeds make the simulated decision-time values reproducible.
np.random.seed(42)
random.seed(42)

In [ ]:
@dataclass
class RoutingPlan:
    """Metrics and decision-support information for one recommendation profile."""

    id: int
    name: str
    routes: List[List[int]]
    total_distance: float
    total_emissions: float
    total_cost: float
    vehicles_used: int
    service_level: float
    explanation: str
    xai_notes: List[str]
    pros: List[str]
    cons: List[str]
    contextual_factors: Dict[str, str]


@dataclass
class CustomerContext:
    """Human-provided information about a customer."""

    id: int
    name: str
    priority_level: str
    special_requirements: List[str]
    relationship_importance: str
    delivery_preferences: Dict[str, str]
    historical_notes: str


@dataclass
class HumanDecision:
    """Record of one dispatcher's choice among recommendation profiles."""

    timestamp: datetime
    plans_considered: List[int]
    chosen_plan: int
    reasoning: str
    contextual_factors: List[str]
    confidence: float
    time_taken: float


class ExplainableAI:
    """Evaluate supplied routes and explain preference-labeled profiles."""

    def __init__(self, distances, emissions, customer_contexts):
        self.distances = np.asarray(distances, dtype=float)
        self.emissions = np.asarray(emissions, dtype=float)
        if self.distances.ndim != 2 or self.distances.shape[0] != self.distances.shape[1]:
            raise ValueError("The distance matrix must be square.")
        if self.emissions.shape != self.distances.shape:
            raise ValueError("Distance and emissions matrices must have the same shape.")

        self.customer_contexts = {customer.id: customer for customer in customer_contexts}
        self.learning_history = []

    def generate_pareto_plans(self, base_routes, weight_combinations):
        """Evaluate fixed routes and create profiles for supplied preference weights.

        The route metrics are independent of the weights: the weights describe
        recommendation emphasis and do not generate alternative routes.
        """
        if not weight_combinations:
            raise ValueError("At least one preference-weight pair is required.")

        for route in base_routes:
            if len(route) < 2:
                raise ValueError("Each route must contain at least two nodes.")
            if any(node < 0 or node >= self.distances.shape[0] for node in route):
                raise ValueError("A route contains a node outside the matrix dimensions.")

        total_distance = 0.0
        total_emissions = 0.0
        for route in base_routes:
            for from_node, to_node in zip(route, route[1:]):
                total_distance += self.distances[from_node, to_node]
                total_emissions += self.emissions[from_node, to_node]

        total_cost = 2.0 * total_distance
        plans = []

        for plan_id, (w_cost, w_emissions) in enumerate(weight_combinations):
            if w_cost < 0 or w_emissions < 0:
                raise ValueError("Preference weights must be non-negative.")
            if not np.isclose(w_cost + w_emissions, 1.0):
                raise ValueError("Each preference-weight pair must sum to 1.")

            name = self._get_plan_name(w_cost, w_emissions)
            explanation = self._generate_explanation(
                w_cost, w_emissions, total_distance, total_emissions, total_cost
            )
            xai_notes = self._generate_xai_notes(w_cost, w_emissions, base_routes)
            pros, cons = self._generate_pros_cons(w_cost, w_emissions)

            plans.append(
                RoutingPlan(
                    id=plan_id,
                    name=name,
                    routes=[route.copy() for route in base_routes],
                    total_distance=total_distance,
                    total_emissions=total_emissions,
                    total_cost=total_cost,
                    vehicles_used=len(base_routes),
                    service_level=0.95,
                    explanation=explanation,
                    xai_notes=xai_notes,
                    pros=pros,
                    cons=cons,
                    contextual_factors=self._analyze_contextual_factors(base_routes),
                )
            )

        return plans

    def _generate_explanation(
        self, w_cost, w_emissions, distance, emissions, cost
    ):
        """Explain the stated preference without implying that routes differ."""
        if w_cost > w_emissions:
            priority = "This profile emphasizes operating cost."
        elif w_emissions > w_cost:
            priority = "This profile emphasizes emissions."
        else:
            priority = "This profile gives equal emphasis to cost and emissions."

        return (
            f"{priority} The supplied routes have a total distance of {distance:.1f} km, "
            f"modeled operating cost of ${cost:.2f}, and emissions of {emissions:.1f} kg CO2. "
            "The preference weights label the profile; they do not change these route metrics."
        )

    def _generate_xai_notes(self, w_cost, w_emissions, routes):
        """Provide notes that distinguish modeled facts from customer context."""
        notes = []
        if w_cost > w_emissions:
            notes.append("Preference profile: cost receives greater emphasis than emissions.")
        elif w_emissions > w_cost:
            notes.append("Preference profile: emissions receive greater emphasis than cost.")
        else:
            notes.append("Preference profile: cost and emissions receive equal emphasis.")

        for route_index, route in enumerate(routes, start=1):
            customer_nodes = route[1:-1]
            notes.append(
                f"Route {route_index} visits {len(customer_nodes)} customer(s); "
                "the route sequence is supplied as input."
            )
            for node in customer_nodes:
                context = self.customer_contexts.get(node)
                if context is not None and context.priority_level == "high":
                    notes.append(
                        f"Customer {node} ({context.name}) is marked high priority in the context data."
                    )

        return notes

    def _generate_pros_cons(self, w_cost, w_emissions):
        """Describe the intended emphasis and its modeling limitation."""
        if w_cost > w_emissions:
            return (
                ["Makes cost the stated decision priority", "Uses an explicit cost metric"],
                ["Does not produce a different route", "Does not establish a cost–emissions trade-off"],
            )
        if w_emissions > w_cost:
            return (
                ["Makes emissions the stated decision priority", "Reports emissions explicitly"],
                ["Does not produce a different route", "Does not establish a cost–emissions trade-off"],
            )
        return (
            ["States a balanced preference", "Reports both objective metrics"],
            ["Does not produce a different route", "Does not establish a cost–emissions trade-off"],
        )

    def _get_plan_name(self, w_cost, w_emissions):
        """Assign a descriptive name to a preference profile."""
        if w_cost > 0.7:
            return "Cost-Focused Profile"
        if w_emissions > 0.7:
            return "Emissions-Focused Profile"
        if abs(w_cost - w_emissions) < 0.1:
            return "Balanced Profile"
        if w_cost > w_emissions:
            return "Cost-Weighted Profile"
        return "Emissions-Weighted Profile"

    def _analyze_contextual_factors(self, routes):
        """Summarize relevant customer context without treating it as a constraint."""
        priority_customers = []
        strategic_customers = []
        for route in routes:
            for node in route[1:-1]:
                context = self.customer_contexts.get(node)
                if context is not None:
                    if context.priority_level == "high":
                        priority_customers.append(node)
                    if context.relationship_importance == "strategic":
                        strategic_customers.append(node)

        factors = {}
        if priority_customers:
            factors["priority_customers"] = f"High-priority customer IDs: {priority_customers}"
        if strategic_customers:
            factors["strategic_customers"] = f"Strategic customer IDs: {strategic_customers}"
        return factors

    def learn_from_decision(self, decision: HumanDecision):
        """Record a decision for review; this demonstration does not adapt its model."""
        self.learning_history.append(decision)


class HumanDispatcher:
    """Apply a stated dispatcher preference and retain the decision record."""

    def __init__(self, name, experience_years, preferences):
        if preferences not in {"cost_focused", "eco_focused", "balanced"}:
            raise ValueError("Preference must be cost_focused, eco_focused, or balanced.")
        self.name = name
        self.experience_years = experience_years
        self.preferences = preferences
        self.decision_history = []
        self.knowledge_base = {
            "traffic_patterns": {"morning_rush": "7-9 AM", "evening_rush": "4-6 PM"},
            "weather_impact": {"rain": "increase_time_20%", "snow": "increase_time_50%"},
            "customer_insights": {},
        }

    def evaluate_plans(self, plans: List[RoutingPlan], current_context: Dict) -> HumanDecision:
        """Review the profiles and select according to the dispatcher's preference."""
        if not plans:
            raise ValueError("At least one recommendation profile is required.")

        print(f"\n{self.name} is reviewing {len(plans)} recommendation profiles.")
        for plan in plans:
            print(f"\nProfile {plan.id}: {plan.name}")
            print(
                f"  Cost: ${plan.total_cost:.2f}; emissions: "
                f"{plan.total_emissions:.1f} kg CO2; distance: {plan.total_distance:.1f} km"
            )
            print(f"  {plan.explanation}")
            for note in plan.xai_notes:
                print(f"  Note: {note}")
            print(f"  Considerations: {'; '.join(plan.pros[:2])}")
            print(f"  Limitations: {'; '.join(plan.cons[:2])}")

        chosen_plan = self._make_decision(plans, current_context)
        decision = HumanDecision(
            timestamp=datetime.now(),
            plans_considered=[plan.id for plan in plans],
            chosen_plan=chosen_plan.id,
            reasoning=self._generate_reasoning(chosen_plan, current_context),
            contextual_factors=list(current_context.keys()),
            confidence=0.85,
            time_taken=random.uniform(2, 8),
        )
        self.decision_history.append(decision)

        print(f"\nSelected profile: {chosen_plan.id} ({chosen_plan.name})")
        print(f"Reasoning: {decision.reasoning}")
        print(f"Reported confidence: {decision.confidence:.0%}")
        return decision

    def _make_decision(self, plans: List[RoutingPlan], context: Dict) -> RoutingPlan:
        """Select a profile using the stated preference; ties follow input order."""
        if self.preferences == "cost_focused":
            return min(plans, key=lambda plan: plan.total_cost)
        if self.preferences == "eco_focused":
            return min(plans, key=lambda plan: plan.total_emissions)

        for plan in plans:
            if "Balanced" in plan.name:
                return plan
        return plans[len(plans) // 2]

    def _generate_reasoning(self, plan: RoutingPlan, context: Dict) -> str:
        """Summarize the selection criterion and the context considered."""
        if self.preferences == "cost_focused":
            parts = [f"Cost preference selected the lowest reported cost (${plan.total_cost:.2f})."]
        elif self.preferences == "eco_focused":
            parts = [
                f"Emissions preference selected the lowest reported emissions "
                f"({plan.total_emissions:.1f} kg CO2)."
            ]
        else:
            parts = ["Balanced preference selected the balanced recommendation profile."]

        if "weather" in context:
            parts.append(f"Weather context supplied: {context['weather']}.")
        if "traffic" in context:
            parts.append(f"Traffic context supplied: {context['traffic']}.")
        parts.append(f"Dispatcher experience: {self.experience_years} years.")
        return " ".join(parts)


class SymbioticPartnership:
    """Coordinate profile generation, human review, and decision recording."""

    def __init__(self, ai_system: ExplainableAI, human_dispatcher: HumanDispatcher):
        self.ai_system = ai_system
        self.human_dispatcher = human_dispatcher
        self.partnership_history = []
        self.trust_metrics = {
            "agreement_rate": None,
            "confidence_trend": [],
            "decision_quality": None,
        }

    def collaborative_decision_session(self, base_routes, current_context):
        """Run one decision session and return the decision and selected profile."""
        print("\n" + "=" * 60)
        print("HUMAN–AI COLLABORATIVE DECISION SESSION")
        print("=" * 60)
        print(f"Dispatcher: {self.human_dispatcher.name}")
        print(f"Experience: {self.human_dispatcher.experience_years} years")
        print(f"Context supplied: {current_context}")

        weight_combinations = [(0.8, 0.2), (0.5, 0.5), (0.2, 0.8)]
        plans = self.ai_system.generate_pareto_plans(base_routes, weight_combinations)
        print("\nProfiles evaluated (all use the same supplied routes):")
        for plan in plans:
            print(
                f"  {plan.id}: {plan.name} — ${plan.total_cost:.2f}, "
                f"{plan.total_emissions:.1f} kg CO2"
            )

        decision = self.human_dispatcher.evaluate_plans(plans, current_context)
        self.ai_system.learn_from_decision(decision)
        chosen_plan = next(plan for plan in plans if plan.id == decision.chosen_plan)

        self.partnership_history.append(
            {
                "timestamp": decision.timestamp,
                "plans_generated": len(plans),
                "chosen_plan": decision.chosen_plan,
                "human_confidence": decision.confidence,
                "decision_time": decision.time_taken,
                "context": current_context.copy(),
            }
        )
        self._update_trust_metrics()
        return decision, chosen_plan

    def _update_trust_metrics(self):
        """Track confidence while leaving undefined metrics explicitly unavailable."""
        recent_sessions = self.partnership_history[-5:]
        self.trust_metrics["confidence_trend"] = [
            session["human_confidence"] for session in recent_sessions
        ]

    def get_partnership_summary(self) -> Dict:
        """Summarize observed sessions without inventing quality or agreement scores."""
        sessions = self.partnership_history
        return {
            "total_sessions": len(sessions),
            "avg_confidence": float(np.mean([s["human_confidence"] for s in sessions]))
            if sessions
            else 0.0,
            "avg_decision_time": float(np.mean([s["decision_time"] for s in sessions]))
            if sessions
            else 0.0,
            "trust_metrics": self.trust_metrics,
        }

In [ ]:
print("=" * 60)
print("HUMAN–AI DECISION-SUPPORT SETUP")
print("=" * 60)

distances = [
    [0, 12, 18, 15, 22, 25, 20],
    [12, 0, 8, 14, 12, 30, 25],
    [18, 8, 0, 10, 6, 22, 18],
    [15, 14, 10, 0, 8, 18, 15],
    [22, 12, 6, 8, 0, 16, 12],
    [25, 30, 22, 18, 16, 0, 14],
    [20, 25, 18, 15, 12, 14, 0],
]

emissions = [
    [0, 10, 15, 12, 20, 28, 18],
    [10, 0, 6, 12, 10, 25, 20],
    [15, 6, 0, 8, 5, 18, 14],
    [12, 12, 8, 0, 6, 15, 12],
    [20, 10, 5, 6, 0, 12, 10],
    [28, 25, 18, 15, 12, 0, 11],
    [18, 20, 14, 12, 10, 11, 0],
]

customer_contexts = [
    CustomerContext(
        id=1,
        name="TechCorp",
        priority_level="high",
        special_requirements=["morning_delivery", "fragile_items"],
        relationship_importance="strategic",
        delivery_preferences={"time_window": "8-10 AM", "packaging": "eco-friendly"},
        historical_notes="Key strategic partner; sensitive to delays.",
    ),
    CustomerContext(
        id=2,
        name="RetailMart",
        priority_level="medium",
        special_requirements=["standard_delivery"],
        relationship_importance="regular",
        delivery_preferences={"time_window": "flexible", "packaging": "standard"},
        historical_notes="Regular customer with flexible scheduling.",
    ),
    CustomerContext(
        id=3,
        name="GreenStore",
        priority_level="high",
        special_requirements=["eco_certified", "carbon_neutral"],
        relationship_importance="strategic",
        delivery_preferences={"time_window": "9-11 AM", "vehicle": "electric_only"},
        historical_notes="Sustainability-focused partner.",
    ),
    CustomerContext(
        id=4,
        name="OfficeSupply",
        priority_level="low",
        special_requirements=["bulk_delivery"],
        relationship_importance="occasional",
        delivery_preferences={"time_window": "afternoon", "packaging": "standard"},
        historical_notes="Occasional bulk orders; flexible on timing.",
    ),
    CustomerContext(
        id=5,
        name="FoodChain",
        priority_level="high",
        special_requirements=["temperature_control", "time_sensitive"],
        relationship_importance="strategic",
        delivery_preferences={"time_window": "6-8 AM", "packaging": "refrigerated"},
        historical_notes="Food-service partner with time-sensitive deliveries.",
    ),
    CustomerContext(
        id=6,
        name="FashionHub",
        priority_level="medium",
        special_requirements=["premium_service"],
        relationship_importance="regular",
        delivery_preferences={"time_window": "10 AM-12 PM", "packaging": "premium"},
        historical_notes="Retailer with premium-service expectations.",
    ),
]

print("Customer context records:")
for context in customer_contexts:
    print(
        f"  {context.name} (ID {context.id}): {context.priority_level} priority, "
        f"{context.relationship_importance} relationship"
    )

base_routes = [
    [0, 1, 3, 5, 0],
    [0, 2, 4, 6, 0],
]

print(f"\nSupplied route structure: {len(base_routes)} routes")
for route_index, route in enumerate(base_routes, start=1):
    node_names = ["Depot" if node == 0 else f"C{node}" for node in route]
    print(f"  Route {route_index}: {' -> '.join(node_names)}")

ai_system = ExplainableAI(distances, emissions, customer_contexts)
human_dispatcher = HumanDispatcher(
    name="Sarah Chen",
    experience_years=12,
    preferences="balanced",
)
partnership = SymbioticPartnership(ai_system, human_dispatcher)

print("\nDecision-support components are ready.")
print(f"  AI component: route-metric evaluation and explanations")
print(f"  Dispatcher: {human_dispatcher.name}, {human_dispatcher.experience_years} years of experience")
print(f"  Stated preference: {human_dispatcher.preferences}")

In [ ]:
print("\n" + "=" * 60)
print("COLLABORATIVE DECISION SESSION")
print("=" * 60)

current_context = {
    "weather": "clear",
    "traffic": "moderate",
    "time_of_day": "morning",
    "special_events": "none",
    "driver_availability": "full",
}

decision, chosen_plan = partnership.collaborative_decision_session(
    base_routes, current_context
)

print("\nSession result")
print(f"  Selected profile: {chosen_plan.name}")
print(f"  Total cost: ${chosen_plan.total_cost:.2f}")
print(f"  Total emissions: {chosen_plan.total_emissions:.1f} kg CO2")
print(f"  Total distance: {chosen_plan.total_distance:.1f} km")
print(f"  Vehicles represented: {chosen_plan.vehicles_used}")
print(f"  Illustrative service-level assumption: {chosen_plan.service_level:.0%}")
print(f"  Dispatcher confidence: {decision.confidence:.0%}")
print(f"  Simulated decision time: {decision.time_taken:.1f} minutes")

In [ ]:
print("\n" + "=" * 60)
print("MULTIPLE SESSIONS: DECISION RECORDS")
print("=" * 60)

scenarios = [
    {
        "context": {
            "weather": "rain",
            "traffic": "heavy",
            "time_of_day": "morning",
            "special_events": "none",
        },
        "description": "Rainy morning with heavy traffic",
    },
    {
        "context": {
            "weather": "clear",
            "traffic": "light",
            "time_of_day": "afternoon",
            "special_events": "none",
        },
        "description": "Clear afternoon with light traffic",
    },
    {
        "context": {
            "weather": "snow",
            "traffic": "moderate",
            "time_of_day": "evening",
            "special_events": "holidays",
        },
        "description": "Snowy evening during a holiday period",
    },
]

for session_number, scenario in enumerate(scenarios, start=2):
    print(f"\n--- Session {session_number}: {scenario['description']} ---")
    session_decision, session_plan = partnership.collaborative_decision_session(
        base_routes, scenario["context"]
    )
    print(f"Selected profile: {session_plan.name}")
    print(
        f"Cost: ${session_plan.total_cost:.2f}; "
        f"emissions: {session_plan.total_emissions:.1f} kg CO2"
    )
    print(
        f"Reported confidence: {session_decision.confidence:.0%}; "
        f"simulated decision time: {session_decision.time_taken:.1f} minutes"
    )

summary = partnership.get_partnership_summary()
print("\nSession summary")
print(f"  Recorded sessions: {summary['total_sessions']}")
print(f"  Mean reported confidence: {summary['avg_confidence']:.1%}")
print(f"  Mean simulated decision time: {summary['avg_decision_time']:.1f} minutes")
print("  Agreement rate: not measured in this example")
print("  Decision quality: not measured in this example")
print("  Learning: decisions are recorded but do not update the recommendation model")
print(f"  Recent confidence observations: {summary['trust_metrics']['confidence_trend']}")

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(18, 12))
fig.suptitle("Human–AI Decision-Support Analysis", fontsize=16, fontweight="bold")

# Compare the objective values of the preference profiles.
ax1 = axes[0, 0]
sample_plans = ai_system.generate_pareto_plans(
    base_routes,
    [
        (0.9, 0.1),
        (0.8, 0.2),
        (0.7, 0.3),
        (0.6, 0.4),
        (0.5, 0.5),
        (0.4, 0.6),
        (0.3, 0.7),
        (0.2, 0.8),
        (0.1, 0.9),
    ],
)

pareto_indices = []
for i, plan in enumerate(sample_plans):
    dominated = any(
        other.total_cost <= plan.total_cost
        and other.total_emissions <= plan.total_emissions
        and (
            other.total_cost < plan.total_cost
            or other.total_emissions < plan.total_emissions
        )
        for j, other in enumerate(sample_plans)
        if i != j
    )
    if not dominated:
        pareto_indices.append(i)

costs = [plan.total_cost for plan in sample_plans]
emissions_values = [plan.total_emissions for plan in sample_plans]
is_frontier = [i in pareto_indices for i in range(len(sample_plans))]
ax1.scatter(
    costs,
    emissions_values,
    c=["green" if value else "red" for value in is_frontier],
    s=[110 if value else 70 for value in is_frontier],
    alpha=0.75,
    label="Non-dominated profile",
)
ax1.scatter([], [], c="red", s=70, alpha=0.75, label="Dominated profile")
for index, plan in enumerate(sample_plans):
    ax1.annotate(
        f"Profile {index + 1}",
        (costs[index], emissions_values[index]),
        xytext=(5, 5 + 13 * (index % 5)),
        textcoords="offset points",
        fontsize=8,
    )
ax1.set_xlabel("Modeled operating cost (USD)")
ax1.set_ylabel("Emissions (kg CO2)")
ax1.set_title("Cost–Emissions Objective Values")
ax1.legend()
ax1.grid(True, alpha=0.3)

# Confidence is a recorded illustrative input, not a trust or quality measure.
ax2 = axes[0, 1]
sessions = list(range(1, len(partnership.partnership_history) + 1))
confidences = [
    session["human_confidence"] for session in partnership.partnership_history
]
ax2.plot(sessions, confidences, "bo-", linewidth=2, markersize=7)
ax2.set_xlabel("Session number")
ax2.set_ylabel("Reported confidence (fraction)")
ax2.set_title("Recorded Dispatcher Confidence")
ax2.set_ylim(0, 1)
ax2.grid(True, alpha=0.3)

ax3 = axes[0, 2]
decision_times = [
    session["decision_time"] for session in partnership.partnership_history
]
ax3.hist(decision_times, bins=5, alpha=0.75, edgecolor="black", color="skyblue")
ax3.set_xlabel("Simulated decision time (minutes)")
ax3.set_ylabel("Number of sessions")
ax3.set_title("Illustrative Decision-Time Distribution")
ax3.grid(True, alpha=0.3)

ax4 = axes[1, 0]
chosen_ids = [
    session["chosen_plan"] for session in partnership.partnership_history
]
plan_counts = [chosen_ids.count(i) for i in range(3)]
plan_labels = ["Cost-focused", "Balanced", "Emissions-focused"]
bars = ax4.bar(plan_labels, plan_counts, alpha=0.75)
ax4.set_ylabel("Recorded selections")
ax4.set_title("Preference-Profile Selections")
ax4.tick_params(axis="x", rotation=20)
ax4.grid(True, axis="y", alpha=0.3)
for bar, count in zip(bars, plan_counts):
    ax4.text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height(),
        str(count),
        ha="center",
        va="bottom",
    )

ax5 = axes[1, 1]
chosen_plans = [
    next(plan for plan in sample_plans if plan.id == plan_id)
    for plan_id in chosen_ids
]
if chosen_plans:
    ax5.scatter(
        [plan.total_cost for plan in chosen_plans],
        [plan.total_emissions for plan in chosen_plans],
        c="purple",
        s=80,
        alpha=0.75,
        label="Selected profile",
    )
    ax5.legend()
else:
    ax5.text(0.5, 0.5, "No recorded selections", ha="center", va="center")
ax5.set_xlabel("Modeled operating cost (USD)")
ax5.set_ylabel("Emissions (kg CO2)")
ax5.set_title("Selected Profiles: Objective Values")
ax5.grid(True, alpha=0.3)

ax6 = axes[1, 2]
ax6.axis("off")
ax6.set_title("Metrics Not Defined by This Example")
ax6.text(
    0.05,
    0.75,
    "Agreement rate: not measured\n\n"
    "Decision quality: not measured\n\n"
    "Model adaptation: not implemented\n\n"
    "Confidence and decision time are recorded for illustration only.",
    transform=ax6.transAxes,
    va="top",
    fontsize=11,
)

plt.tight_layout()
plt.show()

In [ ]:
print("\n" + "=" * 60)
print("INTERPRETING THE DECISION-SUPPORT EXAMPLE")
print("=" * 60)

print("\n1. ROUTE METRICS")
print("   Distance, cost, and emissions are calculated from the supplied route arcs.")
print("   The cost assumption is a constant rate of $2 per kilometer.")

print("\n2. RECOMMENDATION PROFILES")
print("   Preference weights label the profiles; they do not alter the route sequence.")
print("   Therefore, the profiles do not represent distinct optimized routing plans.")

print("\n3. CONTEXT AND HUMAN JUDGMENT")
print("   Customer, weather, and traffic information can inform dispatcher reasoning.")
print("   Those factors are not encoded as route constraints or metric adjustments here.")

print("\n4. DECISION RECORDS")
print(f"   Sessions recorded: {summary['total_sessions']}")
print(f"   Mean reported confidence: {summary['avg_confidence']:.1%}")
print(f"   Mean simulated decision time: {summary['avg_decision_time']:.1f} minutes")
print("   These records alone do not establish trust, agreement, or decision quality.")

print("\n5. MODEL LIMITS")
print("   No route search, vehicle-capacity constraints, time windows, or fleet choices are modeled.")
print("   The learning history records choices but does not modify future recommendations.")
print("   A genuine cost–emissions frontier requires candidate routes with differing objective values.")

## Key takeaways

1. **Separate measurements from preferences.** The distance, cost, and emissions values are calculated from the route data. Preference labels communicate what a dispatcher says matters; they do not change those measurements.
2. **Interpret a Pareto plot carefully.** The Pareto comparison is meaningful only when candidates can differ in objective values. Here, all profiles use the same routes, so the display cannot show a genuine cost–emissions trade-off.
3. **Treat context according to how it enters the model.** Customer priorities, weather, and traffic are available to support human reasoning, but they are not route constraints or metric adjustments in this example.
4. **Distinguish records from evidence of performance.** Confidence and decision time are illustrative session records. Without an independently defined agreement criterion or outcome-based quality measure, they do not quantify trust or decision quality.
5. **Describe the learning loop accurately.** The example stores human decisions for review; it does not update the recommendation logic. Recording feedback and learning from feedback are different operations.

## When human–AI decision support is useful

This workflow is useful when a human decision-maker needs transparent calculations alongside contextual judgment and retains responsibility for the final choice. For operational use, the model would need suitable candidate routes, explicit feasibility rules, validated cost and emissions inputs, and clearly defined measures for evaluating decisions. Those additions are outside the scope of this demonstration.

## Summary

The notebook illustrates how route metrics, explanations, customer context, and a human decision record can be organized in a decision-support workflow. Its prescribed routes make the example easy to inspect, but also limit what can be concluded: the profiles are not separate route solutions, no Pareto trade-off is demonstrated, and no learning or trust improvement is measured. Keeping these boundaries explicit is essential for interpreting the calculations and communicating recommendations responsibly.